# V10.2: SSL Pretrain + Text Fine-tune

**The cross-domain approach.** Same BraTS2021 data, different task.

| Stage | Task | Data | Labels | Text |
|-------|------|------|--------|------|
| **SSL Pretrain** | Masked reconstruction | BraTS2021 (1251) | **NO** | NO |
| **Fine-tune** | Segmentation | BraTS2020 (295) | YES | **YES** |

Backbone learns brain anatomy but NOT tumor segmentation.
Text fills the knowledge gap: 'where are the tumors?'

This mirrors TextBraTS's SwinUNETR approach:
- SwinUNETR: ImageNet (general vision) → text guides tumor segmentation
- V10.2: BraTS2021 SSL (brain anatomy) → text guides tumor segmentation


In [ ]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi 2>/dev/null || echo 'No GPU'
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob, threading
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0: break
        if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else: raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

# BraTS2020 + TextBraTS
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)

# BraTS2021
BRATS2021_LOCAL = '/content/BraTS2021'
BRATS2021_ZIP = os.path.join(DRIVE_BASE, 'BraTS2021_archive.zip')
if os.path.exists(os.path.join(BRATS2021_LOCAL, 'train')):
    print(f'BraTS2021 already extracted')
else:
    print('Extracting BraTS2021...')
    os.makedirs('/content/brats2021_tmp', exist_ok=True)
    !unzip -q {BRATS2021_ZIP} -d /content/brats2021_tmp/
    !tar xf /content/brats2021_tmp/BraTS2021_Training_Data.tar -C /content/brats2021_tmp/
    os.makedirs(BRATS2021_LOCAL, exist_ok=True)
    !mv /content/brats2021_tmp/BraTS2021_* {BRATS2021_LOCAL}/ 2>/dev/null; true
    !python scripts/prepare_brats.py --input {BRATS2021_LOCAL} --output {BRATS2021_LOCAL}
    !rm -rf /content/brats2021_tmp

def sync_and_tag(tag):
    lc = os.path.join(REPO_DIR, 'checkpoints')
    if not os.path.exists(lc): return
    for f in glob.glob(os.path.join(lc, '*.pth')):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    b = os.path.join(lc, 'best.pth')
    if os.path.exists(b): shutil.copy2(b, os.path.join(DRIVE_CKPT, f'best_{tag}.pth'))
    l = os.path.join(lc, 'last.pth')
    if os.path.exists(l): shutil.copy2(l, os.path.join(DRIVE_CKPT, f'last_{tag}.pth'))
    print(f'Synced to {DRIVE_CKPT}')

print('Setup complete')


## Stage 1: SSL Pretrain (Masked Reconstruction, No Labels)

200 epochs on BraTS2021 images. 50% patch masking, L1 reconstruction loss.


In [ ]:
import os, shutil
os.chdir(REPO_DIR)
os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

SSL_BEST = os.path.join(DRIVE_CKPT, 'best_ssl.pth')
if os.path.exists(SSL_BEST):
    print(f'SSL pretrain already done: {SSL_BEST}')
    print('Skip to Stage 2')
else:
    # Check for resume
    resume = ''
    ssl_last = os.path.join(DRIVE_CKPT, 'last_ssl.pth')
    if os.path.exists(ssl_last):
        os.makedirs('checkpoints', exist_ok=True)
        shutil.copy2(ssl_last, 'checkpoints/last_ssl.pth')
        resume = '--resume checkpoints/last_ssl.pth'
        print(f'Resuming SSL from Drive')
    else:
        print('Starting SSL pretrain fresh')

    !python -u scripts/pretrain_ssl.py \
        --data-dir /content/BraTS2021 \
        --epochs 200 \
        --batch-size 4 \
        --lr 1e-4 \
        --mask-ratio 0.5 \
        --embed-dim 48 \
        {resume}

    # Sync SSL checkpoints to Drive
    for name in ['best_ssl.pth', 'last_ssl.pth']:
        src = os.path.join('checkpoints', name)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DRIVE_CKPT, name))
    print('SSL pretrain complete!')


In [ ]:
# === Emergency Sync ===
import shutil, glob, os, subprocess
lc = os.path.join(REPO_DIR, 'checkpoints')
fs = glob.glob(os.path.join(lc, '*.pth'))
if fs:
    for f in sorted(fs):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    subprocess.run(['sync'], check=True)
    print(f'{len(fs)} files synced')
else: print('No checkpoints')


## Stage 2: Fine-tune with Text on BraTS2020

Load SSL-pretrained encoder weights into TextMamba3D, then fine-tune
with SeqCA text fusion on BraTS2020+TextBraTS.


In [ ]:
import os, glob, torch
os.chdir(REPO_DIR)
os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

# Load SSL checkpoint and convert to TextMamba3D format
SSL_CKPT = os.path.join(DRIVE_CKPT, 'best_ssl.pth')
assert os.path.exists(SSL_CKPT), f'SSL checkpoint not found: {SSL_CKPT}'

# Create a pseudo-checkpoint that train.py can load
ssl_state = torch.load(SSL_CKPT, map_location='cpu', weights_only=False)
encoder_state = ssl_state['encoder']

# Build full model state dict with SSL encoder + random init for rest
from models.textmamba3d import TextMamba3D
model = TextMamba3D(
    img_size=(128,128,128), embed_dim=48, depths=[2,2,2,2],
    text_embed_dim=256, use_mamba3=True, headdim=48,
    fusion_type='seqca',
)
full_state = model.state_dict()

# Replace img_encoder weights with SSL pretrained
loaded = 0
for k, v in encoder_state.items():
    full_key = f'img_encoder.{k}'
    if full_key in full_state and full_state[full_key].shape == v.shape:
        full_state[full_key] = v
        loaded += 1
print(f'Loaded {loaded} encoder params from SSL checkpoint')

# Save as train.py-compatible checkpoint
ckpt_dir = os.path.join(REPO_DIR, 'checkpoints')
os.makedirs(ckpt_dir, exist_ok=True)
for f in glob.glob(os.path.join(ckpt_dir, '*.pth')):
    os.remove(f)

ssl_resume = os.path.join(ckpt_dir, 'ssl_init.pth')
torch.save({
    'model': full_state,
    'epoch': -1,
    'best_dice': 0,
    'best_dice_no_text': 0,
}, ssl_resume)
print(f'Saved SSL-initialized checkpoint: {ssl_resume}')

# Fine-tune
print('Stage 2: Fine-tune with text')
!python -u train.py \
    --config configs/autoresearch/V10.2_ssl_pretrain.yaml \
    --resume "{ssl_resume}" \
    --reset-optimizer \
    --no-text-ratio 0.15 \
    --grad-accum 2

sync_and_tag('V10.2')
print('V10.2 Stage 2 complete!')


## Evaluation


In [ ]:
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_V10.2.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Evaluating: {ckpt}')

CONFIG = 'configs/autoresearch/V10.2_ssl_pretrain.yaml'
for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    print()
    print('=' * 60)
    print(name)
    print('=' * 60)
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', CONFIG, '--checkpoint', ckpt,
           '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(ret.stdout)
    if ret.returncode != 0:
        print(f'ERROR: {ret.stderr[-500:]}')

print()
print('Comparison:')
print('  V5.0  (scratch, SeqCA):     Mean=0.8479, delta=+0.55%')
print('  V8.0  (sup pretrain):       Mean=0.8753, delta=0.00%')
print('  V10.2 (SSL pretrain+text):  Mean=?, delta=?')
print('  TextBraTS (SwinUNETR+IN):   Mean=0.853, delta=+1.5%')
